In [1]:
import numpy as np
import pandas as pd
import time
import os
import sys

# On indique à Python où trouver nos "moteurs"
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import du modèle fait main (Partie 5)
from src.mlp_numpy import MLP
# Imports des modèles Keras (Partie 7)
from src.mlp_keras import creer_modele_base, creer_modele_optimise

print("=== Chargement des données ===")
X_train = np.load("../donnees/features/X_train.npy")
X_val   = np.load("../donnees/features/X_val.npy")
X_test  = np.load("../donnees/features/X_test.npy")
y_train = np.load("../donnees/features/y_train.npy")
y_val   = np.load("../donnees/features/y_val.npy")
y_test  = np.load("../donnees/features/y_test.npy")

n_entree = X_train.shape[1]
n_classes = len(np.unique(y_train))

print(f"Dimensions : {n_entree} entrées → {n_classes} classes")

I0000 00:00:1774603705.500461   98102 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774603705.501102   98102 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774603707.416868   98102 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774603707.417310   98102 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


=== Chargement des données ===
Dimensions : 300 entrées → 3 classes


In [2]:
print("=== Lancement du Benchmark (NumPy vs Keras) ===")
resultats_comparaison = []

# ---------------------------------------------------------
# 1. Modèle NumPy (Fait main)
# ---------------------------------------------------------
print("1/3 - Entraînement du modèle NumPy (100 époques)...")
np.random.seed(42)
modele_numpy = MLP(n_entree=n_entree, n_classes=n_classes, taux_apprentissage=0.1)

debut = time.time()
modele_numpy.fit(X_train, y_train, X_val, y_val, n_epoques=100, taille_batch=32, verbose=False)
temps_numpy = time.time() - debut
acc_numpy = modele_numpy.calculer_accuracy(X_test, y_test)

resultats_comparaison.append({
    "Modèle": "NumPy (Fait main)",
    "Optimiseur": "SGD (lr=0.1)",
    "Époques": 100,
    "Temps (sec)": round(temps_numpy, 2),
    "Test Accuracy": f"{acc_numpy*100:.2f}%"
})

# ---------------------------------------------------------
# 2. Modèle Keras Base (Équivalent strict)
# ---------------------------------------------------------
print("2/3 - Entraînement du modèle Keras de base (100 époques)...")
modele_keras_base = creer_modele_base(n_entree, n_classes, taux_apprentissage=0.1)

debut = time.time()
modele_keras_base.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=32, verbose=0)
temps_keras_base = time.time() - debut
_, acc_keras_base = modele_keras_base.evaluate(X_test, y_test, verbose=0)

resultats_comparaison.append({
    "Modèle": "Keras (Base)",
    "Optimiseur": "SGD (lr=0.1)",
    "Époques": 100,
    "Temps (sec)": round(temps_keras_base, 2),
    "Test Accuracy": f"{acc_keras_base*100:.2f}%"
})

# ---------------------------------------------------------
# 3. Modèle Keras Optimisé (Adam, Dropout, BatchNorm)
# ---------------------------------------------------------
print("3/3 - Entraînement du modèle Keras optimisé (50 époques)...")
modele_keras_opti = creer_modele_optimise(n_entree, n_classes)

debut = time.time()
# L'optimiseur Adam est si puissant qu'on réduit à 50 époques
modele_keras_opti.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32, verbose=0)
temps_keras_opti = time.time() - debut
_, acc_keras_opti = modele_keras_opti.evaluate(X_test, y_test, verbose=0)

resultats_comparaison.append({
    "Modèle": "Keras (Optimisé)",
    "Optimiseur": "Adam + BatchNorm",
    "Époques": 50,
    "Temps (sec)": round(temps_keras_opti, 2),
    "Test Accuracy": f"{acc_keras_opti*100:.2f}%"
})

# ---------------------------------------------------------
# Affichage du Livrable
# ---------------------------------------------------------
df_comparaison = pd.DataFrame(resultats_comparaison)
print("\n=== TABLEAU COMPARATIF : NUMPY vs KERAS ===")
display(df_comparaison)

=== Lancement du Benchmark (NumPy vs Keras) ===
1/3 - Entraînement du modèle NumPy (100 époques)...
2/3 - Entraînement du modèle Keras de base (100 époques)...


E0000 00:00:1774603725.396305   98102 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3/3 - Entraînement du modèle Keras optimisé (50 époques)...

=== TABLEAU COMPARATIF : NUMPY vs KERAS ===


,Modèle,Optimiseur,Époques,Temps (sec),Test Accuracy
0,NumPy (Fait main),SGD (lr=0.1),100,8.05,98.22%
1,Keras (Base),SGD (lr=0.1),100,18.14,48.89%
2,Keras (Optimisé),Adam + BatchNorm,50,12.48,100.00%


In [3]:
# On sauvegarde les modèles Keras dans le dossier prévu
os.makedirs("../modeles", exist_ok=True)
modele_keras_base.save("../modeles/keras_base.keras")
modele_keras_opti.save("../modeles/keras_optimise.keras")

print("Modèles Keras sauvegardés avec succès dans le dossier 'modeles/'.")

Modèles Keras sauvegardés avec succès dans le dossier 'modeles/'.
